# Session 8 — LLM API Access with the OpenAI SDK

This notebook is OpenAI SDK-first. We will use the same mental model to reference OpenAI, GitHub Models, and Ollama, then briefly compare that to Claude's Python SDK.

## Learning Goals

- use the OpenAI SDK from Python
- understand Chat Completions vs Responses API
- see how GitHub Models and Ollama can fit the same client pattern
- understand that provider-compatible does not mean provider-identical

In [14]:
import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from pydantic import BaseModel, ConfigDict
import anthropic

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_ORG_ID = os.getenv("OPENAI_ORG_ID")
OPENAI_PROJECT_ID = os.getenv("OPENAI_PROJECT_ID")
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1")

print("OpenAI key present:", bool(OPENAI_API_KEY))
print("OpenAI org ID present:", bool(OPENAI_ORG_ID))
print("OpenAI project ID present:", bool(OPENAI_PROJECT_ID))
print("GitHub token present:", bool(GITHUB_TOKEN))
print("Anthropic key present:", bool(ANTHROPIC_API_KEY))
print("Ollama base URL:", OLLAMA_BASE_URL)

OpenAI key present: True
OpenAI org ID present: True
OpenAI project ID present: True
GitHub token present: True
Anthropic key present: True
Ollama base URL: http://localhost:11434/v1


## Helper Functions

The functions below keep the live API calls small and explicit. If you do not have the required key or local service, skip the relevant cells.

In [15]:
def make_openai_client(api_key=None, base_url=None):
    kwargs = {}
    if api_key:
        kwargs["api_key"] = api_key
    if base_url:
        kwargs["base_url"] = base_url
    if api_key == OPENAI_API_KEY and not base_url:
        if OPENAI_ORG_ID:
            kwargs["organization"] = OPENAI_ORG_ID
        if OPENAI_PROJECT_ID:
            kwargs["project"] = OPENAI_PROJECT_ID
    return OpenAI(**kwargs)


def require_env(name: str):
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f"Missing required environment variable: {name}")
    return value

## 1. Chat Completions API

Chat Completions uses an explicit `messages` array. It is still common in tutorials and production code.

In [16]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

client = make_openai_client(api_key=require_env("OPENAI_API_KEY"))

chat_response = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0,
    messages=[
        {"role": "system", "content": "You are a concise teaching assistant."},
        {"role": "user", "content": "In one sentence, what is retrieval-augmented generation?"},
    ],
)

display(Markdown(chat_response.choices[0].message.content))

Retrieval-augmented generation is a technique that combines information retrieval and natural language generation to enhance the quality and relevance of generated text by incorporating external knowledge from a database or document set.

## 2. Responses API

The Responses API separates `instructions` from `input`, which is often cleaner for application code.

In [17]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

client = make_openai_client(api_key=require_env("OPENAI_API_KEY"))

responses_result = client.responses.create(
    model="gpt-4o-mini",
    instructions="You are a concise teaching assistant.",
    input="In one sentence, what is retrieval-augmented generation?",
)

display(Markdown(responses_result.output_text))

Retrieval-augmented generation is a method that enhances natural language generation by integrating external information from a knowledge base or database to improve the accuracy and relevance of the generated content.

### Quick Comparison

- Chat Completions: explicit message history
- Responses API: cleaner top-level structure
- Both are useful; Session 8 emphasizes the Responses API for newer app workflows

## 3. Structured Output

Applications often need machine-readable output, not just free text.

In [5]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

client = make_openai_client(api_key=require_env("OPENAI_API_KEY"))

structured = client.responses.create(
    model="gpt-4o-mini",
    input="Policy question: What is RAG and when should we use it?",
    text={
        "format": {
            "type": "json_schema",
            "name": "session8_summary",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "topic": {"type": "string"},
                    "use_case": {"type": "string"}
                },
                "required": ["topic", "use_case"],
                "additionalProperties": False
            }
        }
    },
)

display(Markdown(f"```json\n{structured.output_text}\n```"))

```json
{"topic":"RAG (Red, Amber, Green) Status Indicator","use_case":"RAG is used in project management to quickly communicate the status of a project or its elements, helping stakeholders understand performance at a glance. It should be used in status reports, project updates, risk assessments, and when prioritizing tasks or resources."}
```

### Why Add Pydantic?

The JSON Schema example above is useful because it shows the structure we want the model to follow. In a Python app, though, we often want a **validated Python object** rather than a raw JSON string.

Pydantic helps because it turns the model output into typed fields we can use directly in application code. That reduces manual JSON parsing, makes validation errors more visible, and gives us a clearer contract between the model and the rest of the program.

In [6]:
class Session8Summary(BaseModel):
    model_config = ConfigDict(extra="forbid")

    topic: str
    use_case: str

### Pattern A: `responses.parse(...)` with Pydantic

Use this when your application is Python-first and you want the SDK to return a typed object directly. This is the most ergonomic pattern because the client handles the parsing step for you.

In [7]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

client = make_openai_client(api_key=require_env("OPENAI_API_KEY"))

parsed_response = client.responses.parse(
    model="gpt-4o-mini",
    instructions="Return structured teaching notes.",
    input="Explain what RAG is and name one situation where we should use it.",
    text_format=Session8Summary,
)

parsed_summary = parsed_response.output_parsed

display(Markdown(
    f"### Parsed Pydantic object\n"
    f"- **topic:** {parsed_summary.topic}\n"
    f"- **use_case:** {parsed_summary.use_case}"
))
parsed_summary

### Parsed Pydantic object
- **topic:** RAG (Red, Amber, Green) Status Reporting
- **use_case:** Project Management Status Updates

Session8Summary(topic='RAG (Red, Amber, Green) Status Reporting', use_case='Project Management Status Updates')

### Pattern B: `responses.create(...)` and then validate with Pydantic

Use this when you still want the lower-level `responses.create(...)` flow, or when you want to keep access to the raw response object and validate the JSON afterward. This pattern is also helpful when you are gradually upgrading existing code.

In [8]:
# Requires OPENAI_API_KEY
# Skip this cell if you do not have live API access.

client = make_openai_client(api_key=require_env("OPENAI_API_KEY"))

validated_response = client.responses.create(
    model="gpt-4o-mini",
    instructions="Return structured teaching notes.",
    input="Explain what chunking is and give one reason it matters in RAG.",
    text={
        "format": {
            "type": "json_schema",
            "name": "session8_summary_with_pydantic_validation",
            "strict": True,
            "schema": Session8Summary.model_json_schema(),
        }
    },
)

validated_summary = Session8Summary.model_validate_json(validated_response.output_text)

display(Markdown(
    f"### Validated after `responses.create(...)`\n"
    f"- **topic:** {validated_summary.topic}\n"
    f"- **use_case:** {validated_summary.use_case}"
))
validated_summary

### Validated after `responses.create(...)`
- **topic:** Chunking
- **use_case:** Chunking assists in Improving Retrieval-Augmented Generation (RAG) performance by enhancing information processing.

Session8Summary(topic='Chunking', use_case='Chunking assists in Improving Retrieval-Augmented Generation (RAG) performance by enhancing information processing.')

### Choosing Between The Patterns

- Keep the explicit JSON Schema pattern when you want the structure to be highly visible and portable across languages.
- Prefer `responses.parse(...)` when you want the cleanest Python application code with typed validated objects.
- Prefer `responses.create(...)` plus Pydantic validation when you want raw response access, existing code compatibility, or explicit control over the validation step.

## 4. Provider Swap Pattern

The OpenAI SDK can also be used with compatible endpoints. This is useful for GitHub Models and Ollama.

In [9]:
provider_examples = {
    "openai": {
        "api_key_env": "OPENAI_API_KEY",
        "base_url": None,
        "model": "gpt-4o-mini",
    },
    "github_models": {
        "api_key_env": "GITHUB_TOKEN",
        "base_url": "https://models.inference.ai.azure.com",
        "model": "gpt-5-mini",
    },
    "ollama": {
        "api_key_env": None,
        "base_url": OLLAMA_BASE_URL,
        "model": "llama3.2:latest",
    },
}

provider_examples

{'openai': {'api_key_env': 'OPENAI_API_KEY',
  'base_url': None,
  'model': 'gpt-4o-mini'},
 'github_models': {'api_key_env': 'GITHUB_TOKEN',
  'base_url': 'https://models.inference.ai.azure.com',
  'model': 'gpt-5-mini'},
 'ollama': {'api_key_env': None,
  'base_url': 'http://localhost:11434/v1',
  'model': 'llama3.2:latest'}}

In [ ]:
# This demonstrates the compatibility pattern. Live execution still depends on access.
# Important: GitHub Models supports Chat Completions on its OpenAI-compatible endpoint,
# but not the Responses API, so the client setup is similar while the method call differs.

for provider_name, provider in provider_examples.items():
    provider_api_key = os.getenv(provider["api_key_env"]) if provider["api_key_env"] else "ollama"
    provider_client = make_openai_client(api_key=provider_api_key, base_url=provider["base_url"])
    print(provider_name)
    print(provider["base_url"])
    print(provider["model"])

    if provider_name == "github_models":
        result = provider_client.chat.completions.create(
            model=provider["model"],
            messages=[
                {"role": "system", "content": "You are a concise assistant."},
                {"role": "user", "content": "Give one sentence explaining embeddings."},
            ],
        )
        display(Markdown(result.choices[0].message.content))
    else:
        result = provider_client.responses.create(
            model=provider["model"],
            instructions="You are a concise assistant.",
            input="Give one sentence explaining embeddings.",
        )
        display(Markdown(result.output_text))

## 5. Minimal Claude Comparison

Claude uses a different Python SDK and message structure. The main point here is contrast, not building a second full workflow.

In [13]:
# Requires ANTHROPIC_API_KEY
# Skip this cell if you do not have Claude access.

anthropic_client = anthropic.Anthropic(api_key=require_env("ANTHROPIC_API_KEY"))

claude_response = anthropic_client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=120,
    system="You are a concise teaching assistant.",
    messages=[
        {"role": "user", "content": "In one sentence, what is chunking in RAG?"}
    ],
)

display(Markdown(claude_response.content[0].text))

Chunking in RAG is the process of splitting source documents into smaller, manageable pieces of text so they can be efficiently indexed, retrieved, and passed to the language model as relevant context.

## Recap

- We used the OpenAI SDK as the main teaching interface.
- Chat Completions and Responses API solve similar tasks with different shapes.
- GitHub Models and Ollama can often reuse the same client pattern.
- Claude is relevant, but it uses a different SDK and should be treated as a comparison point here.

The next notebook uses these ideas to build a minimal RAG pipeline over PDF documents.